# Aula 10 · Dataclasses

Esta aula apresenta o [capítulo 10 do site](https://lacouth.github.io/python_telecom-site/unidade5-modelagem/10-dataclasses/). A ideia central: **um equipamento é um tipo**.
Com uma dataclass você diz ao Python quais campos um equipamento tem — e ele passa
a reclamar na hora quando um campo é esquecido ou digitado errado, em vez de
deixar o erro virar um número errado no relatório.

**Ao fim da aula você consegue:**

1. criar uma dataclass com campos, valor padrão e métodos;
2. percorrer uma lista de objetos com os padrões de laço de sempre;
3. construir objetos a partir de dados de arquivo, convertendo os tipos na entrada.

**Roteiro:** 🔥 aquecimento · 📟 chamado · 1. o problema · 2. `@dataclass` · 3.
valor padrão · 4. métodos · 5. lista de objetos · 6. do arquivo para o objeto · 7.
o campo digitado errado · 📟 resolvendo o chamado · 🚪 antes de sair

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a solução e rode a célula `confere` logo abaixo dela —
  ✅ quer dizer que acertou, ❌ mostra o que ainda falta. A dica e uma solução
  estão recolhidas: tente antes de abrir.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler agora (usa coisas que só veremos mais tarde).
import math


def _mostra(argumentos):
    return ", ".join(repr(a) for a in argumentos)


def _igual(veio, esperado):
    if isinstance(esperado, float) and isinstance(veio, (int, float)):
        return math.isclose(veio, esperado, abs_tol=1e-9)
    return veio == esperado


def confere(funcao, casos):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if _igual(veio, esperado):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if _igual(valor, esperado):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

## 🔥 Aquecimento — da aula passada

Sem rodar nada: o que este trecho imprime?

```python
linha = "ts=2026-03-02T14:07:44 host=RADIO-01 sev=warn"
campos = {}
for par in linha.split():
    chave, _, valor = par.partition("=")
    campos[chave] = valor
print(campos["sev"], len(campos))
print("-- MARK --".partition("="))
```

<details>
<summary><b>Resposta</b></summary>

Imprime `warn 3` e `('-- MARK --', '', '')`. Cada par vira uma chave do
dicionário; e o `partition` sem o separador não dá erro — devolve o texto inteiro no
primeiro pedaço e os outros dois vazios.

</details>

## 📟 O chamado de hoje

> **Chamado #1011 — NOC Maré Net**
>
> *"Estagiário, o relatório mensal diz que temos 63 portas em serviço, mas a
> equipe de campo contou 73. O cadastro vive numa lista de dicionários, montada à
> mão por várias pessoas. Acho que alguém digitou alguma coisa errada — e o script
> não reclamou de nada. Descobre onde está o erro e faz o cadastro reclamar da
> próxima vez?"*

No fim da aula você carrega o cadastro em objetos — e o erro aparece sozinho.

## 1. O problema das listas paralelas e dos dicionários soltos

Listas paralelas ligam os campos só pela posição; dicionários aceitam qualquer
chave, inclusive a digitada errado. Nos dois casos, o engano **não dá mensagem**.

📖 [capítulo 10 · O problema das listas paralelas e dos dicionários soltos](https://lacouth.github.io/python_telecom-site/unidade5-modelagem/10-dataclasses/#o-problema-das-listas-paralelas-e-dos-dicionarios-soltos)

> 💡 **Pense assim: as duas folhas soltas.**
>
> Imagine os nomes dos alunos numa folha e os telefones em outra, na mesma ordem. Se
> alguém põe a folha dos nomes em ordem alfabética e esquece a outra, cada nome
> passa a ter o telefone de outra pessoa — e nada no papel avisa. É o defeito das
> listas paralelas. O dicionário resolve a ligação, mas aceita qualquer rótulo, até
> um escrito errado.

In [ ]:
# 📦 dados prontos — só rode esta célula
nomes = ["OLT-CENTRO-01", "ONU-SUL-4512", "OLT-NORTE-02"]
ips = ["10.0.1.10", "10.0.3.47", "10.0.2.10"]
equipamento = {"nome": "OLT-CENTRO-01", "portas": 16}

**✍️ Passo 1.** Imprima `nomes[1], ips[1]`. Depois faça `nomes.sort()` e imprima `nomes[1], ips[1]`
de novo.

In [ ]:
# ✍️ passo 1

**Preveja:** depois de ordenar os nomes, o segundo nome continua com o IP certo?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Antes: `ONU-SUL-4512 10.0.3.47`. Depois: `OLT-NORTE-02 10.0.3.47` — a OLT ficou com o
IP da ONU. Só uma das listas foi ordenada, e a posição, que era a única coisa que
ligava nome e IP, deixou de valer.

</details>

**✍️ Passo 2.** Com o dicionário `equipamento` da célula de dados, imprima
`equipamento.get("porta", 0)` — com `porta` no singular.

In [ ]:
# ✍️ passo 2

**Preveja:** dá erro, ou devolve alguma coisa?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Devolve `0`, sem erro nenhum. Para o dicionário, `"porta"` é só uma chave que não
existe — e o `get` com padrão faz exatamente o que se pediu. Um total de portas
calculado assim sai errado por meses.

</details>

## 2. `@dataclass`

`@dataclass` em cima de uma `class` transforma uma lista de campos (`nome: tipo`)
num **tipo** novo. Cada equipamento criado é um **objeto** desse tipo, e os campos
se leem com **ponto**: `olt.nome`.

📖 [capítulo 10 · `@dataclass`](https://lacouth.github.io/python_telecom-site/unidade5-modelagem/10-dataclasses/#dataclass)

> 💡 **Pense assim: o modelo da ficha e as fichas preenchidas.**
>
> A classe é o **modelo da ficha em branco**, impresso na gráfica: diz quais campos
> existem (nome, IP, portas) e em que ordem. Cada objeto é **uma ficha preenchida**:
> `Equipamento("OLT-CENTRO-01", "10.0.1.10", 16)` pega uma ficha nova e preenche.
> Com um modelo, dá para preencher quantas fichas quiser — e ninguém consegue
> inventar um campo que não está impresso no modelo.

**✍️ Passo 3.** Escreva `from dataclasses import dataclass`. Depois, `@dataclass` numa linha e,
embaixo, `class Equipamento:` com três campos recuados: `nome: str`, `ip: str` e
`portas: int`. Crie `olt = Equipamento("OLT-CENTRO-01", "10.0.1.10", 16)` e imprima
`olt`, `olt.nome` e `olt.portas + 8`.

In [ ]:
# ✍️ passo 3

**Preveja:** o que o `print(olt)` mostra?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`Equipamento(nome='OLT-CENTRO-01', ip='10.0.1.10', portas=16)`, depois
`OLT-CENTRO-01` e `24`. A dataclass monta sozinha um `print` legível, com o nome de
cada campo. E `olt.portas` é um número de verdade, que entra na conta.

</details>

**✍️ Passo 4.** Crie outro com os campos pelo nome, em outra ordem:
`outra = Equipamento(ip="10.0.1.10", nome="OLT-CENTRO-01", portas=16)`, e imprima
`outra == olt`.

In [ ]:
# ✍️ passo 4

**Preveja:** dois objetos criados separadamente, com os mesmos valores, são iguais?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`True`: a dataclass compara **campo a campo**. E passando pelo nome, a ordem deixa
de importar — a chamada fica mais longa, mas se lê sozinha.

</details>

**✍️ Passo 5.** Tente criar um equipamento esquecendo as portas:
`Equipamento("OLT-CENTRO-01", "10.0.1.10")`.

In [ ]:
# ✍️ passo 5

**Preveja:** o Python aceita um equipamento sem portas?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`TypeError: ... missing 1 required positional argument: 'portas'`. Diferente do
dicionário, que aceitaria qualquer coisa, a dataclass sabe quais campos um
equipamento **precisa** ter — e reclama na criação.

</details>

### 🎯 Sua vez — Uma dataclass sua

Crie a dataclass `Porta`, com os campos `numero` (inteiro) e `estado` (texto, com
valor padrão `"down"`). As duas funções da célula de conferência só existem para
testar a sua classe.

In [ ]:
from dataclasses import dataclass


@dataclass
class Porta:
    # sua solução aqui
    pass

In [ ]:
def cria_sem_estado(numero):
    porta = Porta(numero)
    return porta.numero, porta.estado


def cria_com_estado(numero, estado):
    porta = Porta(numero, estado)
    return porta.numero, porta.estado


confere(cria_sem_estado, [((3,), (3, "down"))])
confere(cria_com_estado, [((7, "up"), (7, "up"))])

<details>
<summary><b>💡 Dica</b></summary>

Dois campos dentro da classe, um por linha: `numero: int` e `estado: str = "down"`.
O de valor padrão vem por último.

</details>

## 3. Valor padrão

Um campo pode ter valor padrão, como um parâmetro de função — e, como nas funções,
os campos com padrão vêm **depois** dos sem padrão.

📖 [capítulo 10 · Valor padrão](https://lacouth.github.io/python_telecom-site/unidade5-modelagem/10-dataclasses/#valor-padrao)

**✍️ Passo 6.** Reescreva a `Equipamento` acrescentando um quarto campo, `em_servico: bool = True`.
Crie `reserva = Equipamento("OLT-NORTE-02", "10.0.2.10", 8, em_servico=False)` e
imprima `reserva` e `Equipamento("X", "1.1.1.1", 1).em_servico`.

In [ ]:
# ✍️ passo 6

**Preveja:** quem não informa `em_servico` fica com qual valor?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

O `reserva` mostra `em_servico=False`, e o equipamento criado sem informar fica com
`True` — o padrão. Se o campo com padrão viesse **antes** de `portas`, o Python
recusaria a classe inteira (`non-default argument ... follows default argument`).

</details>

## 4. Métodos: a função que mora junto do dado

Uma conta que só faz sentido para um enlace pode morar **dentro** da classe: é um
**método**. O primeiro parâmetro, `self`, é **o objeto da vez** — quem o passa é o
próprio Python.

📖 [capítulo 10 · Métodos: a função que mora junto do dado](https://lacouth.github.io/python_telecom-site/unidade5-modelagem/10-dataclasses/#metodos-a-funcao-que-mora-junto-do-dado)

> 📡 **Na rede: potência recebida e sensibilidade.**
>
> `potencia_rx` é a potência que **chega** ao receptor (*rx* é a abreviação de
> recepção; *tx*, de transmissão). A `sensibilidade` é a menor potência que o
> receptor ainda entende — nas ONUs da Maré Net, −27 dBm. A **margem** é a folga
> entre as duas: se é positiva, o enlace **fecha** (funciona); se é negativa, o sinal
> chega fraco demais. Veja
> [Atenuação, sensibilidade e margem](https://lacouth.github.io/python_telecom-site/unidade0-primeiros-passos/rede-marenet/#atenuacao-sensibilidade-e-margem).

> 💡 **Pense assim: “este aqui”.**
>
> Numa fila de conferência de documentos, o fiscal diz a cada pessoa: "mostre **o
> seu** documento". A frase é a mesma para todo mundo, mas "o seu" muda conforme
> quem está na frente dele. O `self` é esse "o seu": o método é escrito uma vez só,
> e o `self` passa a ser o objeto que está sendo atendido naquela chamada.

**✍️ Passo 7.** Escreva a dataclass `Enlace` com `nome: str`, `potencia_rx: float` e
`sensibilidade: float = -27.0`. Dentro dela, recuado, o método `def margem(self):`
que devolve `round(self.potencia_rx - self.sensibilidade, 2)`. Crie
`sul = Enlace("ONU-SUL-4512", -21.4)` e `leste = Enlace("ONU-LESTE-77", -28.1)` e
imprima `sul.margem()` e `leste.margem()`.

In [ ]:
# ✍️ passo 7

**Preveja:** por que a chamada `sul.margem()` tem parênteses vazios, se o método tem o
parâmetro `self`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`5.6` e `-1.1`. Em `sul.margem()`, o Python passa o `sul` como `self` sozinho; em
`leste.margem()`, passa o `leste`. Dentro do método, `self.potencia_rx` é a potência
**daquele** enlace.

</details>

**✍️ Passo 8.** Imprima `Enlace.margem(sul)`.

In [ ]:
# ✍️ passo 8

**Preveja:** dá o mesmo resultado de `sul.margem()`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Dá `5.6`, o mesmo. Não há mágica: o método é uma função comum que recebe o enlace
como primeiro argumento. `sul.margem()` é só a forma curta — a mesma de
`texto.upper()`, que você usa desde a Aula 02.

</details>

> ⚠️ **Armadilha.** Esquecer o `self` na definição (`def margem():`) ou dentro dela
> (`potencia_rx - sensibilidade`). No primeiro caso, a chamada dá `TypeError` dizendo
> que o método recebeu um argumento a mais; no segundo, `NameError`, porque os campos
> só existem **no objeto**, e o caminho até eles é o `self.`.

### 🎯 Sua vez — Precisa de visita?

A equipe de campo visita todo enlace com margem **menor que 3 dB**. Complete o
método `precisa_visita` da `Enlace` abaixo.

In [ ]:
from dataclasses import dataclass


@dataclass
class Enlace:
    nome: str
    potencia_rx: float
    sensibilidade: float = -27.0

    def margem(self):
        return round(self.potencia_rx - self.sensibilidade, 2)

    def precisa_visita(self):
        # sua solução aqui
        pass

In [ ]:
confere(Enlace.precisa_visita, [
    ((Enlace("ONU-SUL-4512", -21.4),), False),
    ((Enlace("ONU-CENTRO-9", -25.0),), True),
    ((Enlace("ONU-LESTE-77", -28.1),), True),
    ((Enlace("ONU-NORTE-5", -24.0),), False),
])

<details>
<summary><b>💡 Dica</b></summary>

Use o outro método: `self.margem() < 3`. Uma comparação já é `True` ou `False` —
dá para devolvê-la direto.

</details>

## 5. Lista de objetos

Um inventário vira uma **lista de equipamentos** — e os padrões da Unidade 1
continuam iguais, com `e.portas` no lugar de `e["portas"]`.

📖 [capítulo 10 · Lista de objetos](https://lacouth.github.io/python_telecom-site/unidade5-modelagem/10-dataclasses/#lista-de-objetos)

In [ ]:
# 📦 dados prontos — só rode esta célula
from dataclasses import dataclass


@dataclass
class Equipamento:
    nome: str
    tipo: str
    portas: int
    em_servico: bool = True


inventario = [
    Equipamento("OLT-CENTRO-01", "OLT", 16),
    Equipamento("ONU-SUL-4512", "ONU", 1),
    Equipamento("OLT-NORTE-02", "OLT", 8, em_servico=False),
    Equipamento("OLT-SUL-03", "OLT", 24),
]

**✍️ Passo 9.** Com o `inventario` da célula de dados, calcule o total de portas **em serviço**:
`total = 0` e, no laço, **se** `e.em_servico`, `total = total + e.portas`. Imprima
`total`.

In [ ]:
# ✍️ passo 9

**Preveja:** que número sai?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`41`: 16 + 1 + 24 — a OLT-NORTE-02 está fora de serviço. É o acumulador da Aula 03,
e o `if` do filtro da Aula 06, com o ponto no lugar dos colchetes.

</details>

**✍️ Passo 10.** Escreva `def portas_do_equipamento(e): return e.portas` e imprima
`sorted(inventario, key=portas_do_equipamento, reverse=True)[0].nome`.

In [ ]:
# ✍️ passo 10

**Preveja:** o que o `sorted` devolve — uma lista de nomes ou de objetos?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Uma lista de **objetos**, ordenada pelas portas; o `[0]` é o objeto com mais portas,
e o `.nome` pega o nome dele: `OLT-SUL-03`. É o `key=` da Aula 05, com uma função
que devolve um campo do objeto.

</details>

### 🎯 Sua vez — Os enlaces para visitar

Escreva `para_visitar(enlaces)`, que recebe uma lista de `Enlace` (a classe do 🎯
anterior) e devolve a lista dos **nomes** dos que precisam de visita.

In [ ]:
def para_visitar(enlaces):
    # sua solução aqui
    pass

In [ ]:
confere(para_visitar, [
    (([Enlace("A", -21.4), Enlace("B", -25.0), Enlace("C", -28.1)],), ["B", "C"]),
    (([Enlace("D", -20.0)],), []),
    (([],), []),
])

<details>
<summary><b>💡 Dica</b></summary>

É o filtro de sempre, com `e.precisa_visita()` no `if` e `e.nome` no `append`.

</details>

## 6. Do arquivo para o objeto

O dado vem de fora como texto, JSON ou CSV. O lugar certo para construir os objetos
é a **entrada** — e é ali que se convertem os tipos, **uma vez só**. Daí em diante,
todo o programa recebe objetos prontos.

📖 [capítulo 10 · Do arquivo para o objeto](https://lacouth.github.io/python_telecom-site/unidade5-modelagem/10-dataclasses/#do-arquivo-para-o-objeto)

In [ ]:
# 📦 dados prontos — só rode esta célula
import json

texto = """
[
  {"nome": "OLT-CENTRO-01", "tipo": "OLT", "portas": "16", "em_servico": true},
  {"nome": "ONU-SUL-4512", "tipo": "ONU", "portas": "1", "em_servico": false}
]
"""

**✍️ Passo 11.** Crie `carregados = []` e percorra `json.loads(texto)`. Para cada dicionário `d`,
crie um `Equipamento` passando cada campo pelo nome — `nome=d["nome"]`,
`tipo=d["tipo"]`, `portas=int(d["portas"])`, `em_servico=d["em_servico"]` — e faça
`append`. Imprima `carregados[0]` e `carregados[0].portas + carregados[1].portas`.

In [ ]:
# ✍️ passo 11

**Preveja:** por que o `int(...)` nas portas?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Porque no JSON as portas vieram como **texto** (`"16"`, entre aspas). Sem o `int`,
a soma daria `"161"` — concatenação. Com a conversão feita na construção, sai `17`,
e ninguém mais no programa precisa lembrar de converter.

</details>

### 🎯 Sua vez — De uma linha de CSV para um Enlace

Escreva `de_linha(linha)`, que recebe uma linha no formato `"nome,potencia"` (como
`"ONU-SUL-4512,-21.4"`) e devolve um `Enlace` com a potência **como número**.

In [ ]:
def de_linha(linha):
    # sua solução aqui
    pass

In [ ]:
confere(de_linha, [
    (("ONU-SUL-4512,-21.4",), Enlace("ONU-SUL-4512", -21.4)),
    (("ONU-LESTE-77,-28.1",), Enlace("ONU-LESTE-77", -28.1)),
])

<details>
<summary><b>💡 Dica</b></summary>

`nome, _, potencia = linha.partition(",")` e depois
`Enlace(nome, float(potencia))`.

</details>

## 7. Atributo digitado errado

E o defeito do começo da aula, agora com uma dataclass.

📖 [capítulo 10 · Atributo digitado errado](https://lacouth.github.io/python_telecom-site/unidade5-modelagem/10-dataclasses/#atributo-digitado-errado)

**✍️ Passo 12.** Pegue um equipamento do inventário, `olt = inventario[0]`, e imprima `olt.porta` —
no singular.

In [ ]:
# ✍️ passo 12

**Preveja:** dá o `0` silencioso do dicionário?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`AttributeError: 'Equipamento' object has no attribute 'porta'. Did you mean:
'portas'?` O Python sabe quais campos um `Equipamento` tem e reclama **na linha do
erro** — com sugestão de correção. O engano que no dicionário virava um zero no
relatório agora não passa.

</details>

## 📟 Resolvendo o chamado

Rode as células abaixo: o cadastro como ele está hoje (lista de dicionários, montada
à mão) e a dataclass que ele deveria respeitar.

In [ ]:
# 📦 dados prontos — só rode esta célula
cadastro = [
    {"nome": "OLT-CENTRO-01", "tipo": "OLT", "portas": 16},
    {"nome": "OLT-SUL-03", "tipo": "OLT", "portas": 24},
    {"nome": "ONU-SUL-4512", "tipo": "ONU", "portas": 1},
    {"nome": "OLT-LESTE-04", "tipo": "OLT", "porta": 10},
    {"nome": "SWITCH-NORTE-02", "tipo": "SWITCH", "portas": 22},
]

In [ ]:
# O total do relatório mensal, do jeito que o script faz hoje:
total = 0
for d in cadastro:
    total = total + d.get("portas", 0)
print(total)

In [ ]:
from dataclasses import dataclass


@dataclass
class Equipamento:
    nome: str
    tipo: str
    portas: int
    em_servico: bool = True

### 🎯 Sua vez — O cadastro que reclama

Escreva `carrega(cadastro)`, que devolve **dois valores**:

1. a lista de `Equipamento` construídos a partir dos dicionários (campo a campo,
   com `d["portas"]` — **sem** `get`);
2. a lista dos **nomes** dos dicionários que não puderam virar equipamento porque
   falta algum campo (`KeyError`).

Use `try/except KeyError` **dentro** do laço, como na Aula 08.

In [ ]:
def carrega(cadastro):
    # sua solução aqui
    pass

In [ ]:
confere(carrega, [
    ((cadastro,), ([Equipamento("OLT-CENTRO-01", "OLT", 16), Equipamento("OLT-SUL-03", "OLT", 24),
                    Equipamento("ONU-SUL-4512", "ONU", 1),
                    Equipamento("SWITCH-NORTE-02", "SWITCH", 22)],
                   ["OLT-LESTE-04"])),
    (([],), ([], [])),
])

<details>
<summary><b>💡 Dica</b></summary>

Duas listas vazias antes do laço. Dentro do `try`, construa o `Equipamento` com
`nome=d["nome"], tipo=d["tipo"], portas=d["portas"]` e faça `append`; no
`except KeyError`, guarde `d["nome"]` na outra lista.

</details>

**Resposta ao chamado:** a `OLT-LESTE-04` foi cadastrada com `"porta"` em vez de
`"portas"`, e o `get("portas", 0)` a contou como zero — são as 10 portas que faltavam
(63 no relatório, 73 em campo). Construindo equipamentos com `d["portas"]`, o erro
aparece na hora, com o nome do culpado.

## 🚪 Antes de sair

Responda de cabeça, sem rodar.

**1.** Numa dataclass, `olt.porta` (com o campo chamado `portas`) dá:
a) `0`  b) `None`  c) `AttributeError`  d) `KeyError`

<details>
<summary><b>Resposta da 1</b></summary>

**c** — e a mensagem ainda sugere `'portas'`. O `KeyError` é do dicionário.

</details>

**2.** Em `sul.margem()`, quem é o `self` dentro do método?
a) a classe `Enlace`  b) o objeto `sul`  c) nada, os parênteses estão vazios
d) o primeiro campo

<details>
<summary><b>Resposta da 2</b></summary>

**b**. O Python passa o objeto da vez como `self`.

</details>

**3.** Por que converter os tipos (`int(...)`) **na construção** do objeto?
a) é obrigatório na dataclass  b) para converter uma vez só e o resto do programa
receber o tipo certo  c) porque o JSON não tem números  d) para o `print` ficar
bonito

<details>
<summary><b>Resposta da 3</b></summary>

**b**. O tipo escrito no campo (`portas: int`) não é conferido pelo Python — quem
garante é a conversão na entrada.

</details>

## 🏠 Para casa

- [Lista 10](https://lacouth.github.io/python_telecom-site/listas/lista10/) —
  dataclasses, com testes automáticos no Colab.
- Releia o [capítulo 10 do site](https://lacouth.github.io/python_telecom-site/unidade5-modelagem/10-dataclasses/).
- Projeto: veja [Organizando o projeto em arquivos](https://lacouth.github.io/python_telecom-site/projeto/organizacao/).
- **Na próxima aula:** mini-teste sobre esta aula (dataclass, método e `self`).